In [124]:
library(tidyr)
library(dplyr)
library(ggplot2)
library(patchwork)
library(data.table)

source("/mnt/lareaulab/reliscu/code/ggplot_theme.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/enrichment_fxns.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/gene_mapping_fxns.R")

theme_set(default_theme())

Loading required package: future


Attaching package: ‘reshape2’


The following object is masked from ‘package:tidyr’:

    smiths


The following objects are masked from ‘package:data.table’:

    dcast, melt




In [125]:
options(repr.plot.width=20, repr.plot.height=8, repr.plot.res=150)

In [126]:
plot_expr_over_samples <- function(expr, gene_list, plot_title) {
    gene_expr <- expr[expr$Gene %in% gene_list,]
    rownames(gene_expr) <- gene_expr$Gene
    ctype_expr_t <- t.data.frame(gene_expr[,-1])

    corr_mat <- cor(ctype_expr_t)
    diag(corr_mat) <- NA
    mean_pairwise_corr <- mean(corr_mat, na.rm=T) 
    median_pairwise_corr <- median(corr_mat, na.rm=TRUE)

    df <- cbind(data.frame(Sample=colnames(expr)[-1]), ctype_expr_t)

    df_long <- df %>%
        pivot_longer(
            cols=-Sample,           # Keep Sample as x-axis
            names_to="Cell_type",   # Column names become a new "Cell_type" column
            values_to="Expr"  # Values become a new "Expression" column
        )

    plot_sub <- paste(
        "Mean pairwise corr:", round(mean_pairwise_corr, 2), 
        "\nMedian pairwise corr:", round(median_pairwise_corr, 2)
    )

    p <- ggplot(data=df_long, aes(x=Sample, y=Expr, color=Cell_type, group=Cell_type)) +
        geom_line(size=0.5) +
        theme(
            plot.title=element_text(size=20, hjust=0.5),
            plot.subtitle=element_text(size=18, hjust=0.5),
            panel.grid.major=element_blank(),
            panel.grid.minor=element_blank(),
            legend.text=element_text(size=18),
            axis.title.x=element_text(size=18),
            axis.text.x=element_blank(),
            axis.title.y=element_text(size=18),
            axis.text.y=element_text(size=18)
        ) +
        labs(title=plot_title, subtitle=plot_sub) +
        ylab("Expression") +
        scale_y_continuous(trans="log")

    print(p)
}

In [4]:
data_source <- "GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected"

expr <- fread(paste0("data/", data_source, ".csv"), data.table=FALSE)

colnames(expr)[1] <- "Gene"

## Visualize predefined markers

### Prep markers

In [ ]:
set_source <- "Claude_marker_genes"

marker_genes_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/marker_genes/AI/Claude_cortical_markers_human.RDS")

names(marker_genes_list)

In [137]:
marker_genes_list <- lapply(marker_genes_list, function(x) {
    if (length(x) > 20) {
        x[1:20]
    } else x
})

### Plot

In [ ]:
for (i in seq_along(marker_genes_list)) {
    ctype_genes <- marker_genes_list[[i]]
    plot_title <- names(marker_genes_list)[i] 
    plot_expr_over_samples(expr, ctype_genes, plot_title)
}

## Compare marker genes to cell type enriched module genes 